In [1]:
#%set_env PATH=$PATH:/usr/local/cuda-12/bin
!nvcc --version
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2023 NVIDIA Corporation
Built on Tue_Aug_15_22:02:13_PDT_2023
Cuda compilation tools, release 12.2, V12.2.140
Build cuda_12.2.r12.2/compiler.33191640_0
Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp3mm59u2c".


In [ ]:
%%cuda
#include <stdio.h>
#include <cuda.h>
#include <stdlib.h>
#include <time.h>
#include <iostream>
#include <chrono>

// this function works with any square kernel and implicit bi-dimensional array
__global__ void gpu_convolution(int kernel_size, double *kernel, int image_width, int image_height, double *img, double *img_result)
{
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = index; i < image_width * image_height; i += stride)
    {
        img_result[i] = 0.; // initialize

        int y = i / image_width;
        int x = i % image_width;

        int kernel_rad = kernel_size / 2; // distance from center to edge
        for (int j = -kernel_rad; j <= kernel_rad; j++)
        {
            if (y + j < 0 || y + j >= image_height)
                continue;

            for (int k = -kernel_rad; k <= kernel_rad; k++)
            {
                if (x + k < 0 || x + k >= image_width)
                    continue;

                img_result[i] += kernel[(j + kernel_rad) * kernel_size + k + kernel_rad] * img[(y + j) * image_width + (x + k)];
            }
        }
    }
}

__global__ void gpu_tiled_convolution(int kernel_size, double *kernel, int image_width, int image_height, double *img, double *img_result)
{
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    int kernel_rad = kernel_size / 2; // distance from center to edge

    for (int i = index; i < image_width * image_height; i += stride)
    {
        img_result[i] = 0.; // initialize

        int y = i / image_width;
        int x = i % image_width;

        for (int j = -kernel_rad; j <= kernel_rad; j++)
        {
            for (int k = -kernel_rad; k <= kernel_rad; k++)
            {
                if (x + k < 0 || x + k >= image_width) // asumed row padding, but no column pads
                    continue;

                img_result[i] += kernel[(j + kernel_rad) * kernel_size + k + kernel_rad] * img[(y + j) * image_width + (x + k)];
            }
        }
    }
}

void cpu_convolution(int kernel_size, double *kernel, int image_width, int image_height, double *img, double *img_result)
{
    for (int i = 0; i < image_width * image_height; i++)
    {
        img_result[i] = 0.; // initialize
        int y = i / image_width;
        int x = i % image_width;
        int kernel_rad = kernel_size / 2; // distance from center to edge
        for (int j = -kernel_rad; j <= kernel_rad; j++)
        {
            if (y + j < 0 || y + j >= image_height)
                continue;

            for (int k = -kernel_rad; k <= kernel_rad; k++)
            {
                if (x + k < 0 || x + k >= image_width)
                    continue;

                img_result[i] += kernel[(j + kernel_rad) * kernel_size + k + kernel_rad] * img[(y + j) * image_width + (x + k)];
            }
        }
    }
}

__global__ void gpu_fill_array(int size, double *arr, double val)
{
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = index; i < size; i += stride)
    {
        arr[i] = val;
    }
}

void tiled_convolution(int kernel_size, double *kernel, int image_width, int image_height, double *img, double *img_result, int tile_row_size, int blocks, int threads)
{
    int kernel_rad = kernel_size / 2; // distance from center to edge

    int img_tile_size = (tile_row_size + kernel_rad * 2) * image_width * sizeof(double);
    int pad_size = kernel_rad * image_width * sizeof(double);
    int img_tile_result_size = tile_row_size * image_width * sizeof(double);

    double *img_tile;
    double *img_tile_result;
    cudaMalloc(&img_tile, img_tile_size);
    cudaMalloc(&img_tile_result, img_tile_result_size);

    for (int i = 0; i < image_height; i += tile_row_size)
    {
        // add padding
        if (i == 0)
        {
            gpu_fill_array<<<blocks, threads>>>(kernel_rad * image_width, img_tile, 0);
            cudaMemcpy(img_tile +kernel_rad * image_width , img, img_tile_size - pad_size, cudaMemcpyHostToDevice);
        }
        else if (i == image_height - tile_row_size)
        {
            gpu_fill_array<<<blocks, threads>>>(kernel_rad * image_width, img_tile + (tile_row_size + kernel_rad) * image_width, 0);
            cudaMemcpy(img_tile, img + ((i - kernel_rad) * image_width), img_tile_size - pad_size, cudaMemcpyHostToDevice);
        }
        else
        {
            cudaMemcpy(img_tile, img + ((i - kernel_rad) * image_width), img_tile_size, cudaMemcpyHostToDevice);
        }

        gpu_tiled_convolution<<<blocks, threads>>>(kernel_size, kernel, image_width, tile_row_size, img_tile + image_width * kernel_rad, img_tile_result);
        cudaError_t kernel_err = cudaDeviceSynchronize();
        if (kernel_err != cudaSuccess)
        {
            printf("Kernel execution failed: %s\n", cudaGetErrorString(kernel_err));
            cudaFree(img_tile);
            cudaFree(img_tile_result);
            return;
        }
        cudaError_t err = cudaMemcpy(img_result + (i * image_width), img_tile_result, img_tile_result_size, cudaMemcpyDeviceToHost);
        if (err != cudaSuccess)
        {
            printf("CUDA memcpy failed: %s\n", cudaGetErrorString(err));
            break;
        }
    }

    cudaFree(img_tile);
    cudaFree(img_tile_result);
}

int main(int argc, char const *argv[])
{
    int nDevices;
    cudaGetDeviceCount(&nDevices);
    for (int i = 0; i < nDevices; i++)
    {
        cudaDeviceProp prop;
        cudaGetDeviceProperties(&prop, i);
        printf("Device Number: %d\n", i);
        printf("  Device name: %s\n", prop.name);
        printf("  max Blocks Per MultiProcessor: %d\n", prop.maxBlocksPerMultiProcessor);
        printf("  max Threads Per MultiProcessor: %d\n", prop.maxThreadsPerMultiProcessor);
        printf("  max Threads Per Block: %d\n", prop.maxThreadsPerBlock);
        printf("  num SM: %d\n", prop.multiProcessorCount);
        printf("  num bytes sharedMem Per Block: %d\n", prop.sharedMemPerBlock);
        printf("  num bytes sharedMem Per Multiprocessor: %d\n", prop.sharedMemPerMultiprocessor);
        printf("  Memory Clock Rate (KHz): %d\n",
               prop.memoryClockRate);
        printf("  Memory Bus Width (bits): %d\n",
               prop.memoryBusWidth);
        printf("  Peak Memory Bandwidth (GB/s): %f\n\n",
               2.0 * prop.memoryClockRate * (prop.memoryBusWidth / 8) / 1.0e6);
    }

    int img_x = 10000, img_y = 2000, kernel_size = 20;
    double *img; // it's easier to vizualice it as an b/w image, and passing a single array is way easier than a bi-dimensional array
    double *img_result;
    double *kernel; // we don't care about the size, so it's variable

    cudaMallocManaged(&img, img_x * img_y * sizeof(double));
    cudaMallocManaged(&img_result, img_x * img_y * sizeof(double));
    cudaMallocManaged(&kernel, kernel_size * kernel_size * sizeof(double));

    for (int i = 0; i < kernel_size * kernel_size; i++)
        kernel[i] = 0.1;

    gpu_fill_array<<<10, 32>>>(img_x * img_y, img, 1);

    float total_mem_bytes = sizeof(double) * img_x * img_y * 2;
    float bytesPerMiB = 1024.0 * 1024.0;
    printf(" memory allocated: %.1f (MiB)\n\n", total_mem_bytes / bytesPerMiB);

    cudaDeviceSynchronize();
    auto start = std::chrono::system_clock::now();
    cpu_convolution(kernel_size, kernel, img_x, img_y, img, img_result);
    auto ende = std::chrono::system_clock::now();
    auto seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "cpu execution time: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    gpu_convolution<<<10, 64>>>(kernel_size, kernel, img_x, img_y, img, img_result);
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 10 blocks of 64 threads: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    gpu_convolution<<<144, 32>>>(kernel_size, kernel, img_x, img_y, img, img_result);
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 5 blocks of 128 threads: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    gpu_convolution<<<24, 128>>>(kernel_size, kernel, img_x, img_y, img, img_result); // max
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 24 blocks of 128 threads: " << seconds << "\n\n";

start = std::chrono::system_clock::now();
    gpu_convolution<<<96, 32>>>(kernel_size, kernel, img_x, img_y, img, img_result); // max
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 96 blocks of 32 threads: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    gpu_convolution<<<3072, 64>>>(kernel_size, kernel, img_x, img_y, img, img_result); // max
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 40 blocks of 64 threads: " << seconds << "\n\n";

start = std::chrono::system_clock::now();
    gpu_convolution<<<3072, 32>>>(kernel_size, kernel, img_x, img_y, img, img_result); // max
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 40 blocks of 64 threads: " << seconds << "\n\n";


    cudaFree(img);
    cudaFree(img_result);

    img = new double[img_x * img_y];
    img_result = new double[img_x * img_y];

    for (int i = 0; i < img_x * img_y; i++)
    {
        img[i] = 1.;
    }

    //tiling execution
    start = std::chrono::system_clock::now();
    tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 1000, 10, 64);
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu tiled execution time with 10 blocks of 64 threads: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 1000, 5, 128);
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu tiled execution time with 5 blocks of 128 threads: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 1000, 80, 32);
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu tiled execution time with 80 blocks of 32 threads: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 1000, 40, 64);
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu tiled execution time with 40 blocks of 64 threads: " << seconds << "\n\n";

    // cudaFree(kernel);
    return 0;
}

In [ ]:
%%cuda
#include <stdio.h>
#include <cuda.h>
#include <stdlib.h>
#include <time.h>
#include <iostream>
#include <chrono>

// this function works with any square kernel and implicit bi-dimensional array
__global__ void gpu_convolution(int kernel_size, double *kernel, int image_width, int image_height, double *img, double *img_result)
{
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = index; i < image_width * image_height; i += stride)
    {
        img_result[i] = 0.; // initialize

        int y = i / image_width;
        int x = i % image_width;

        int kernel_rad = kernel_size / 2; // distance from center to edge
        for (int j = -kernel_rad; j <= kernel_rad; j++)
        {
            if (y + j < 0 || y + j >= image_height)
                continue;

            for (int k = -kernel_rad; k <= kernel_rad; k++)
            {
                if (x + k < 0 || x + k >= image_width)
                    continue;

                img_result[i] += kernel[(j + kernel_rad) * kernel_size + k + kernel_rad] * img[(y + j) * image_width + (x + k)];
            }
        }
    }
}

__global__ void gpu_tiled_convolution(int kernel_size, double *kernel, int image_width, int image_height, double *img, double *img_result)
{
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    int kernel_rad = kernel_size / 2; // distance from center to edge

    for (int i = index; i < image_width * image_height; i += stride)
    {
        img_result[i] = 0.; // initialize

        int y = i / image_width;
        int x = i % image_width;

        for (int j = -kernel_rad; j <= kernel_rad; j++)
        {
            for (int k = -kernel_rad; k <= kernel_rad; k++)
            {
                if (x + k < 0 || x + k >= image_width) // asumed row padding, but no column pads
                    continue;

                img_result[i] += kernel[(j + kernel_rad) * kernel_size + k + kernel_rad] * img[(y + j) * image_width + (x + k)];
            }
        }
    }
}

void cpu_convolution(int kernel_size, double *kernel, int image_width, int image_height, double *img, double *img_result)
{
    for (int i = 0; i < image_width * image_height; i++)
    {
        img_result[i] = 0.; // initialize
        int y = i / image_width;
        int x = i % image_width;
        int kernel_rad = kernel_size / 2; // distance from center to edge
        for (int j = -kernel_rad; j <= kernel_rad; j++)
        {
            if (y + j < 0 || y + j >= image_height)
                continue;

            for (int k = -kernel_rad; k <= kernel_rad; k++)
            {
                if (x + k < 0 || x + k >= image_width)
                    continue;

                img_result[i] += kernel[(j + kernel_rad) * kernel_size + k + kernel_rad] * img[(y + j) * image_width + (x + k)];
            }
        }
    }
}

__global__ void gpu_fill_array(int size, double *arr, double val)
{
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = index; i < size; i += stride)
    {
        arr[i] = val;
    }
}

void tiled_convolution(int kernel_size, double *kernel, int image_width, int image_height, double *img, double *img_result, int tile_row_size, int blocks, int threads)
{
    int kernel_rad = kernel_size / 2; // distance from center to edge

    int img_tile_size = (tile_row_size + kernel_rad * 2) * image_width * sizeof(double);
    int pad_size = kernel_rad * image_width * sizeof(double);
    int img_tile_result_size = tile_row_size * image_width * sizeof(double);

    double *img_tile;
    double *img_tile_result;
    cudaMalloc(&img_tile, img_tile_size);
    cudaMalloc(&img_tile_result, img_tile_result_size);

    for (int i = 0; i < image_height; i += tile_row_size)
    {
        // add padding
        if (i == 0)
        {
            gpu_fill_array<<<blocks, threads>>>(kernel_rad * image_width, img_tile, 0);
            cudaMemcpy(img_tile +kernel_rad * image_width , img, img_tile_size - pad_size, cudaMemcpyHostToDevice);
        }
        else if (i == image_height - tile_row_size)
        {
            gpu_fill_array<<<blocks, threads>>>(kernel_rad * image_width, img_tile + (tile_row_size + kernel_rad) * image_width, 0);
            cudaMemcpy(img_tile, img + ((i - kernel_rad) * image_width), img_tile_size - pad_size, cudaMemcpyHostToDevice);
        }
        else
        {
            cudaMemcpy(img_tile, img + ((i - kernel_rad) * image_width), img_tile_size, cudaMemcpyHostToDevice);
        }

        gpu_tiled_convolution<<<blocks, threads>>>(kernel_size, kernel, image_width, tile_row_size, img_tile + image_width * kernel_rad, img_tile_result);
        cudaError_t kernel_err = cudaDeviceSynchronize();
        if (kernel_err != cudaSuccess)
        {
            printf("Kernel execution failed: %s\n", cudaGetErrorString(kernel_err));
            cudaFree(img_tile);
            cudaFree(img_tile_result);
            return;
        }
        cudaError_t err = cudaMemcpy(img_result + (i * image_width), img_tile_result, img_tile_result_size, cudaMemcpyDeviceToHost);
        if (err != cudaSuccess)
        {
            printf("CUDA memcpy failed: %s\n", cudaGetErrorString(err));
            break;
        }
    }

    cudaFree(img_tile);
    cudaFree(img_tile_result);
}

int main(int argc, char const *argv[])
{
    int nDevices;
    cudaGetDeviceCount(&nDevices);
    for (int i = 0; i < nDevices; i++)
    {
        cudaDeviceProp prop;
        cudaGetDeviceProperties(&prop, i);
        printf("Device Number: %d\n", i);
        printf("  Device name: %s\n", prop.name);
        printf("  max Blocks Per MultiProcessor: %d\n", prop.maxBlocksPerMultiProcessor);
        printf("  max Threads Per MultiProcessor: %d\n", prop.maxThreadsPerMultiProcessor);
        printf("  max Threads Per Block: %d\n", prop.maxThreadsPerBlock);
        printf("  num SM: %d\n", prop.multiProcessorCount);
        printf("  num bytes sharedMem Per Block: %d\n", prop.sharedMemPerBlock);
        printf("  num bytes sharedMem Per Multiprocessor: %d\n", prop.sharedMemPerMultiprocessor);
        printf("  Memory Clock Rate (KHz): %d\n",
               prop.memoryClockRate);
        printf("  Memory Bus Width (bits): %d\n",
               prop.memoryBusWidth);
        printf("  Peak Memory Bandwidth (GB/s): %f\n\n",
               2.0 * prop.memoryClockRate * (prop.memoryBusWidth / 8) / 1.0e6);
    }

    int img_x = 10000, img_y = 2000, kernel_size = 20;
    double *img; // it's easier to vizualice it as an b/w image, and passing a single array is way easier than a bi-dimensional array
    double *img_result;
    double *kernel; // we don't care about the size, so it's variable

    cudaMallocManaged(&img, img_x * img_y * sizeof(double));
    cudaMallocManaged(&img_result, img_x * img_y * sizeof(double));
    cudaMallocManaged(&kernel, kernel_size * kernel_size * sizeof(double));

    for (int i = 0; i < kernel_size * kernel_size; i++)
        kernel[i] = 0.1;

    gpu_fill_array<<<10, 32>>>(img_x * img_y, img, 1);

    float total_mem_bytes = sizeof(double) * img_x * img_y * 2;
    float bytesPerMiB = 1024.0 * 1024.0;
    printf(" memory allocated: %.1f (MiB)\n\n", total_mem_bytes / bytesPerMiB);

    cudaDeviceSynchronize();
    auto start = std::chrono::system_clock::now();
    //cpu_convolution(kernel_size, kernel, img_x, img_y, img, img_result);
    auto ende = std::chrono::system_clock::now();
    auto seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "cpu execution time: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    gpu_convolution<<<10, 64>>>(kernel_size, kernel, img_x, img_y, img, img_result);
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 10 blocks of 64 threads: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    gpu_convolution<<<144, 32>>>(kernel_size, kernel, img_x, img_y, img, img_result);
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 5 blocks of 128 threads: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    gpu_convolution<<<24, 128>>>(kernel_size, kernel, img_x, img_y, img, img_result); // max
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 24 blocks of 128 threads: " << seconds << "\n\n";

start = std::chrono::system_clock::now();
    gpu_convolution<<<96, 32>>>(kernel_size, kernel, img_x, img_y, img, img_result); // max
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 96 blocks of 32 threads: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    gpu_convolution<<<3072, 64>>>(kernel_size, kernel, img_x, img_y, img, img_result); // max
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 40 blocks of 64 threads: " << seconds << "\n\n";

start = std::chrono::system_clock::now();
    gpu_convolution<<<3072, 32>>>(kernel_size, kernel, img_x, img_y, img, img_result); // max
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 40 blocks of 64 threads: " << seconds << "\n\n";


    cudaFree(img);
    cudaFree(img_result);

    img = new double[img_x * img_y];
    img_result = new double[img_x * img_y];

    for (int i = 0; i < img_x * img_y; i++)
    {
        img[i] = 1.;
    }

    //tiling execution
    start = std::chrono::system_clock::now();
    tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 1000, 10, 64);
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu tiled execution time with 10 blocks of 64 threads: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 1000, 5, 128);
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu tiled execution time with 5 blocks of 128 threads: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 1000, 80, 32);
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu tiled execution time with 80 blocks of 32 threads: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 1000, 40, 64);
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu tiled execution time with 40 blocks of 64 threads: " << seconds << "\n\n";

    // cudaFree(kernel);
    return 0;
}

Device Number: 0
  Device name: Tesla T4
  max Blocks Per MultiProcessor: 16
  max Threads Per MultiProcessor: 1024
  max Threads Per Block: 1024
  num SM: 40
  num bytes sharedMem Per Block: 49152
  num bytes sharedMem Per Multiprocessor: 65536
  Memory Clock Rate (KHz): 5001000
  Memory Bus Width (bits): 256
  Peak Memory Bandwidth (GB/s): 320.064000

 memory allocated: 305.2 (MiB)

3 img values: 1.000000, 1.000000, 1.000000
first 3 result diagonal values: 0.000000, 0.000000, 0.000000
cpu execution time: 5.5e-08

3 img values: 1.000000, 1.000000, 1.000000
first 3 result diagonal values: 10.900000, 11.900000, 14.300000
gpu execution time with 10 blocks of 64 threads: 1.62242

3 img values: 1.000000, 1.000000, 1.000000
first 3 result diagonal values: 10.900000, 11.900000, 14.300000
gpu execution time with 5 blocks of 128 threads: 0.270272

3 img values: 1.000000, 1.000000, 1.000000
first 3 result diagonal values: 10.900000, 11.900000, 14.300000
gpu execution time with 24 blocks of 128 

In [ ]:
%%cuda
#include <stdio.h>
#include <cuda.h>
#include <stdlib.h>
#include <time.h>
#include <iostream>
#include <chrono>
#include <cuda_runtime_api.h>

// this function works with any square kernel and implicit bi-dimensional array
__global__ void gpu_convolution(int kernel_size, double *kernel, int image_width, int image_height, double *img, double *img_result)
{
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = index; i < image_width * image_height; i += stride)
    {
        img_result[i] = 0.; // initialize

        int y = i / image_width;
        int x = i % image_width;

        int kernel_rad = kernel_size / 2; // distance from center to edge
        for (int j = -kernel_rad; j <= kernel_rad; j++)
        {
            if (y + j < 0 || y + j >= image_height)
                continue;

            for (int k = -kernel_rad; k <= kernel_rad; k++)
            {
                if (x + k < 0 || x + k >= image_width)
                    continue;

                img_result[i] += kernel[(j + kernel_rad) * kernel_size + k + kernel_rad] * img[(y + j) * image_width + (x + k)];
            }
        }
    }
}

__global__ void gpu_tiled_convolution(int kernel_size, double *kernel, int image_width, int image_height, double *img, double *img_result)
{
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    int kernel_rad = kernel_size / 2; // distance from center to edge

    for (int i = index; i < image_width * image_height; i += stride)
    {
        img_result[i] = 0.; // initialize

        int y = i / image_width;
        int x = i % image_width;

        for (int j = -kernel_rad; j <= kernel_rad; j++)
        {
            for (int k = -kernel_rad; k <= kernel_rad; k++)
            {
                if (x + k < 0 || x + k >= image_width) // asumed row padding, but no column pads
                    continue;

                img_result[i] += kernel[(j + kernel_rad) * kernel_size + k + kernel_rad] * img[(y + j) * image_width + (x + k)];
            }
        }
    }
}

void cpu_convolution(int kernel_size, double *kernel, int image_width, int image_height, double *img, double *img_result)
{
    for (int i = 0; i < image_width * image_height; i++)
    {
        img_result[i] = 0.; // initialize
        int y = i / image_width;
        int x = i % image_width;
        int kernel_rad = kernel_size / 2; // distance from center to edge
        for (int j = -kernel_rad; j <= kernel_rad; j++)
        {
            if (y + j < 0 || y + j >= image_height)
                continue;

            for (int k = -kernel_rad; k <= kernel_rad; k++)
            {
                if (x + k < 0 || x + k >= image_width)
                    continue;

                img_result[i] += kernel[(j + kernel_rad) * kernel_size + k + kernel_rad] * img[(y + j) * image_width + (x + k)];
            }
        }
    }
}

__global__ void gpu_fill_array(int size, double *arr, double val)
{
    int index = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;

    for (int i = index; i < size; i += stride)
    {
        arr[i] = val;
    }
}

void tiled_convolution(int kernel_size, double *kernel, int image_width, int image_height, double *img, double *img_result, int tile_row_size, int blocks, int threads)
{
    int kernel_rad = kernel_size / 2; // distance from center to edge

    int img_tile_size = (tile_row_size + kernel_rad * 2) * image_width * sizeof(double);
    int pad_size = kernel_rad * image_width * sizeof(double);
    int img_tile_result_size = tile_row_size * image_width * sizeof(double);

    double *img_tile;
    double *img_tile_result;
    cudaMalloc(&img_tile, img_tile_size);
    cudaMalloc(&img_tile_result, img_tile_result_size);

    for (int i = 0; i < image_height; i += tile_row_size)
    {
        // add padding
        if (i == 0)
        {
            gpu_fill_array<<<blocks, threads>>>(kernel_rad * image_width, img_tile, 0);
            cudaMemcpy(img_tile +kernel_rad * image_width , img, img_tile_size - pad_size, cudaMemcpyHostToDevice);
        }
        else if (i == image_height - tile_row_size)
        {
            gpu_fill_array<<<blocks, threads>>>(kernel_rad * image_width, img_tile + (tile_row_size + kernel_rad) * image_width, 0);
            cudaMemcpy(img_tile, img + ((i - kernel_rad) * image_width), img_tile_size - pad_size, cudaMemcpyHostToDevice);
        }
        else
        {
            cudaMemcpy(img_tile, img + ((i - kernel_rad) * image_width), img_tile_size, cudaMemcpyHostToDevice);
        }

        gpu_tiled_convolution<<<blocks, threads>>>(kernel_size, kernel, image_width, tile_row_size, img_tile + image_width * kernel_rad, img_tile_result);
        cudaError_t kernel_err = cudaDeviceSynchronize();
        if (kernel_err != cudaSuccess)
        {
            printf("Kernel execution failed: %s\n", cudaGetErrorString(kernel_err));
            cudaFree(img_tile);
            cudaFree(img_tile_result);
            return;
        }
        cudaError_t err = cudaMemcpy(img_result + (i * image_width), img_tile_result, img_tile_result_size, cudaMemcpyDeviceToHost);
        if (err != cudaSuccess)
        {
            printf("CUDA memcpy failed: %s\n", cudaGetErrorString(err));
            break;
        }
    }

    cudaFree(img_tile);
    cudaFree(img_tile_result);
}

int main(int argc, char const *argv[])
{
    int nDevices;
    cudaGetDeviceCount(&nDevices);
    for (int i = 0; i < nDevices; i++)
    {
        cudaDeviceProp prop;
        cudaGetDeviceProperties(&prop, i);
        printf("Device Number: %d\n", i);
        printf("  Device name: %s\n", prop.name);
        printf("  max Blocks Per MultiProcessor: %d\n", prop.maxBlocksPerMultiProcessor);
        printf("  max Threads Per MultiProcessor: %d\n", prop.maxThreadsPerMultiProcessor);
        printf("  max Threads Per Block: %d\n", prop.maxThreadsPerBlock);
        printf("  num SM: %d\n", prop.multiProcessorCount);
        printf("  num bytes sharedMem Per Block: %d\n", prop.sharedMemPerBlock);
        printf("  num bytes sharedMem Per Multiprocessor: %d\n", prop.sharedMemPerMultiprocessor);
        printf("  Memory Clock Rate (KHz): %d\n",
               prop.memoryClockRate);
        printf("  Memory Bus Width (bits): %d\n",
               prop.memoryBusWidth);
        printf("  Peak Memory Bandwidth (GB/s): %f\n\n",
               2.0 * prop.memoryClockRate * (prop.memoryBusWidth / 8) / 1.0e6);
    }

    int img_x = 10000, img_y = 2000, kernel_size = 20;
    double *img; // it's easier to vizualice it as an b/w image, and passing a single array is way easier than a bi-dimensional array
    double *img_result;
    double *kernel; // we don't care about the size, so it's variable

    cudaMallocManaged(&img, img_x * img_y * sizeof(double));
    cudaMallocManaged(&img_result, img_x * img_y * sizeof(double));
    cudaMallocManaged(&kernel, kernel_size * kernel_size * sizeof(double));

    for (int i = 0; i < kernel_size * kernel_size; i++)
        kernel[i] = 0.1;

    gpu_fill_array<<<10, 32>>>(img_x * img_y, img, 1);

    float total_mem_bytes = sizeof(double) * img_x * img_y * 2;
    float bytesPerMiB = 1024.0 * 1024.0;
    printf(" memory allocated: %.1f (MiB)\n\n", total_mem_bytes / bytesPerMiB);

    cudaDeviceSynchronize();
    auto start = std::chrono::system_clock::now();
    //cpu_convolution(kernel_size, kernel, img_x, img_y, img, img_result);
    auto ende = std::chrono::system_clock::now();
    auto seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "cpu execution time: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    gpu_convolution<<<10, 64>>>(kernel_size, kernel, img_x, img_y, img, img_result);
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 10 blocks of 64 threads: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    gpu_convolution<<<144, 32>>>(kernel_size, kernel, img_x, img_y, img, img_result);
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 5 blocks of 128 threads: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    gpu_convolution<<<24, 128>>>(kernel_size, kernel, img_x, img_y, img, img_result); // max
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 24 blocks of 128 threads: " << seconds << "\n\n";

start = std::chrono::system_clock::now();
    gpu_convolution<<<96, 32>>>(kernel_size, kernel, img_x, img_y, img, img_result); // max
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 96 blocks of 32 threads: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    gpu_convolution<<<3072, 64>>>(kernel_size, kernel, img_x, img_y, img, img_result); // max
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 40 blocks of 64 threads: " << seconds << "\n\n";

start = std::chrono::system_clock::now();
    gpu_convolution<<<3072, 32>>>(kernel_size, kernel, img_x, img_y, img, img_result); // max
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 40 blocks of 64 threads: " << seconds << "\n\n";

    start = std::chrono::system_clock::now();
    gpu_convolution<<<1280, 32>>>(kernel_size, kernel, img_x, img_y, img, img_result); // max
    cudaDeviceSynchronize();
    ende = std::chrono::system_clock::now();
    seconds = std::chrono::duration<double>(ende - start).count();
    printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    std::cout << "gpu execution time with 40 blocks of 64 threads: " << seconds << "\n\n";

    cudaFree(img);
    cudaFree(img_result);

    img = new double[img_x * img_y];
    img_result = new double[img_x * img_y];

    for (int i = 0; i < img_x * img_y; i++)
    {
        img[i] = 1.;
    }

    //tiling execution
    # start = std::chrono::system_clock::now();
    # tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 1000, 10, 64);
    # ende = std::chrono::system_clock::now();
    # seconds = std::chrono::duration<double>(ende - start).count();
    # printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    # printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    # std::cout << "gpu tiled execution time with 10 blocks of 64 threads: " << seconds << "\n\n";

    # start = std::chrono::system_clock::now();
    # tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 1000, 5, 128);
    # ende = std::chrono::system_clock::now();
    # seconds = std::chrono::duration<double>(ende - start).count();
    # printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    # printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    # std::cout << "gpu tiled execution time with 5 blocks of 128 threads: " << seconds << "\n\n";

    # start = std::chrono::system_clock::now();
    # tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 1000, 80, 32);
    # ende = std::chrono::system_clock::now();
    # seconds = std::chrono::duration<double>(ende - start).count();
    # printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    # printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    # std::cout << "gpu tiled execution time with 80 blocks of 32 threads: " << seconds << "\n\n";

    # start = std::chrono::system_clock::now();
    # tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 1000, 40, 64);
    # ende = std::chrono::system_clock::now();
    # seconds = std::chrono::duration<double>(ende - start).count();
    # printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    # printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    # std::cout << "gpu tiled execution time with 40 blocks of 64 threads: " << seconds << "\n\n";

    # start = std::chrono::system_clock::now();
    # tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 625, 10, 64);
    # ende = std::chrono::system_clock::now();
    # seconds = std::chrono::duration<double>(ende - start).count();
    # printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    # printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    # std::cout << "gpu tiled execution time with 40 blocks of 64 threads: " << seconds << "\n\n";

    # start = std::chrono::system_clock::now();
    # tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 625, 5, 128);
    # ende = std::chrono::system_clock::now();
    # seconds = std::chrono::duration<double>(ende - start).count();
    # printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    # printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    # std::cout << "gpu tiled execution time with 5 blocks of 128 threads: " << seconds << "\n\n";

    # start = std::chrono::system_clock::now();
    # tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 625, 80, 32);
    # ende = std::chrono::system_clock::now();
    # seconds = std::chrono::duration<double>(ende - start).count();
    # printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    # printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    # std::cout << "gpu tiled execution time with 80 blocks of 32 threads: " << seconds << "\n\n";

    # start = std::chrono::system_clock::now();
    # tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 625, 40, 64);
    # ende = std::chrono::system_clock::now();
    # seconds = std::chrono::duration<double>(ende - start).count();
    # printf("3 img values: %f, %f, %f\n", img[0], img[1], img[img_x * 2 + 1]);
    # printf("first 3 result diagonal values: %f, %f, %f\n", img_result[0], img_result[1], img_result[img_x * 2 + 1]);
    # std::cout << "gpu tiled execution time with 40 blocks of 64 threads: " << seconds << "\n\n";

    // cudaFree(kernel);
    return 0;
}

/tmp/tmp3mm59u2c/f32974eb-dd2f-4934-ad6c-08afa9a88142/single_file.cu:278:7: error: invalid preprocessing directive #start
  278 |     # start = std::chrono::system_clock::now();
      |       ^~~~~
/tmp/tmp3mm59u2c/f32974eb-dd2f-4934-ad6c-08afa9a88142/single_file.cu:279:7: error: invalid preprocessing directive #tiled_convolution
  279 |     # tiled_convolution(kernel_size, kernel, img_x, img_y, img, img_result, 1000, 10, 64);
      |       ^~~~~~~~~~~~~~~~~
/tmp/tmp3mm59u2c/f32974eb-dd2f-4934-ad6c-08afa9a88142/single_file.cu:280:7: error: invalid preprocessing directive #ende
  280 |     # ende = std::chrono::system_clock::now();
      |       ^~~~
/tmp/tmp3mm59u2c/f32974eb-dd2f-4934-ad6c-08afa9a88142/single_file.cu:281:7: error: invalid preprocessing directive #seconds
  281 |     # seconds = std::chrono::duration<double>(ende - start).count();
      |       ^~~~~~~
/tmp/tmp3mm59u2c/f32974eb-dd2f-4934-ad6c-08afa9a88142/single_file.cu:282:7: error: invalid preprocessing directive #pri

#setup
nvidia T4 GPU 2,560 cores, which means max 80 blocks of 32 threads.

#Design choices
Even if a 2d matrix is usually represented by a bi-dimensional array, it's easier to just use a 1D array and then run the transformations within the classes.
No pad, that would require extra computation power to move the original matrix to a bigger one, (but the actual computation would be cheaper) assuming the complete matrix actually fits inside the GPU unified memory

for tiled execution we assume pad for only rows, as addig pad for columns would be computationally expensive for 1D implicit representation of 2D

tile is customizable with the number of rows, which must be a factor of the number of rows.

#Performance measurements
check the code prints